# Module 27 — Exercise 3: SQLAlchemy 2.0 ORM and N+1 Query Mitigation

In this exercise, you will define declarative models in modern SQLAlchemy 2.0 syntax, perform relationships and joins, and eliminate N+1 query traps using eager loading (`selectinload`).

| Detail | Value |
|---|---|
| **Time** | 40 minutes |
| **Prerequisites** | Module 27 README, Module 11 |



## 1. Declarative Models with `Mapped` and `mapped_column`


In [ ]:
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy import ForeignKey, String, select
from typing import List

class Base(DeclarativeBase):
    pass

class Customer(Base):
    __tablename__ = "customers"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    orders: Mapped[List["Order"]] = relationship(back_populates="customer")

class Order(Base):
    __tablename__ = "orders"
    id: Mapped[int] = mapped_column(primary_key=True)
    total: Mapped[float]
    customer_id: Mapped[int] = mapped_column(ForeignKey("customers.id"))
    customer: Mapped["Customer"] = relationship(back_populates="orders")



# Your turn


### Task 1: Building Query Filters in SQLAlchemy 2.0

Write `get_high_value_orders(session, min_total)` that executes a `select(Order).where(Order.total >= min_total)` query and returns a list of matching `Order` instances.


In [ ]:
# ANSWER 1
def get_high_value_orders(session, min_total: float):
    stmt = select(Order).where(Order.total >= min_total)
    return list(session.scalars(stmt).all())



## Self-Check Harness


In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(engine)

with Session(engine) as session:
    c = Customer(name="Sarah")
    c.orders = [Order(total=45.0), Order(total=120.0), Order(total=310.0)]
    session.add(c)
    session.commit()

with Session(engine) as session:
    high_orders = get_high_value_orders(session, 100.0)
    results = [
        check(len(high_orders) == 2, "Task 1: Correctly filtered orders >= 100.0"),
        check({o.total for o in high_orders} == {120.0, 310.0}, "Task 1: Retrieved matching order totals"),
    ]
    print(f"Summary: {sum(results)}/{len(results)} checks passed.")

